# 정제 모듈 (02_clean)

**입력**: config.yaml에 지정된 입력 파일 (A/B/C/D형 대응)  
**출력**: `master_gl_clean.csv` (원본 컬럼 + 파생변수), `master_gl_subtotal.csv` (분리된 소계행)

처리 순서: STEP 0 로드 → STEP 1 column_map → STEP 2 소계행 → STEP 3 날짜 → STEP 4 금액 → STEP 5 코드변환 → STEP 6 이상치 → 검증 → 저장

> 클라이언트가 달라지면 **config.yaml만 교체**, 노트북 코드는 수정 불필요.

In [57]:
import pandas as pd
import numpy as np
import yaml
import re

In [ ]:
# config 로드 — 이 셀만 수정하면 클라이언트 전환 가능
CONFIG_FILE = 'config_clean.yaml'

with open(CONFIG_FILE, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print(f"config 로드: {cfg['client']}")
print(f"입력 파일:   {cfg['paths']['input_file']}")
print(f"금액 형태:   {cfg['amount']['type']}형")

config 로드: SAP_sample
입력 파일:   master_gl_절대금액 표시.csv
금액 형태:   C형


In [59]:
# STEP 0 — 파일 로드
df_raw = pd.read_csv(
    cfg['paths']['input_file'],
    dtype={col: str for col in cfg['str_cols']},
    encoding=cfg['encoding'],
    low_memory=False
)
df = df_raw.copy()   # 원본 보존
RAW_LEN = len(df)

print(f"로드 완료: {RAW_LEN:,}행 / {df.shape[1]}열")
print(df.dtypes)

로드 완료: 332,103행 / 18열
전표번호            object
회사코드            object
회계연도            object
라인번호            object
전기일             object
증빙일             object
회계월             object
계정코드            object
계정명             object
계정속성(BS/PL)     object
현지통화금액         float64
차대구분            object
통합거래처명          object
원가센터명(부서명)      object
적요              object
전표성격            object
역분개여부           object
역분개참조번호         object
dtype: object


In [60]:
# STEP 1 — column_map 적용 (한글 → 표준 영어)
df = df.rename(columns=cfg['column_map'])

# 매핑 후 한글 컬럼 잔존 여부 확인
unmapped = [c for c in df.columns if any(ord(ch) > 127 for ch in c)]
print("매핑 완료:", list(df.columns))
if unmapped:
    print("⚠️ 한글 컬럼 잔존 (column_map에 추가 필요):", unmapped)

매핑 완료: ['doc_no', 'company_code', 'fisc_year', 'line_no', 'post_date', 'doc_date', 'fisc_month', 'acc_code', 'acc_name', 'acc_type', 'amount', 'dc_indicator', 'counterparty', 'department', 'description', 'doc_type', 'reversal_yn', 'reversal_ref']


In [61]:
# STEP 2 — 소계행 탐지 및 분리
sub_cfg = cfg['subtotal']

mask_subtotal = (
    df[sub_cfg['required_not_null']].isna() |
    df['acc_name'].str.contains('|'.join(sub_cfg['keywords']), case=False, na=False) |
    (df['acc_code'].str.len() < sub_cfg['acc_code_min_len'])
)

df_subtotal = df[mask_subtotal].copy()   # 분리 보존 (삭제 X)
df          = df[~mask_subtotal].copy()

print(f"소계행 분리: {len(df_subtotal):,}행")
print(f"분석 대상:   {len(df):,}행")
print(f"행수 합계:   {len(df) + len(df_subtotal):,} (원본: {RAW_LEN:,}) "
      f"{'✅' if len(df) + len(df_subtotal) == RAW_LEN else '❌ 불일치'}")

소계행 분리: 0행
분석 대상:   332,103행
행수 합계:   332,103 (원본: 332,103) ✅


In [62]:
# STEP 3 — 날짜 파싱 함수 정의 (다형성 지원)
def parse_date_flexible(series):
    """다양한 날짜 형태를 datetime으로 변환
    지원 형태:
      - 구분자형: 2022-04-24 / 2022.04.24 / 2022/04/24
      - 8자리 숫자형: 20220424
      - 엑셀 시리얼형: 44675 (정수)
      - 한글형: 2022년 4월 24일
    """
    # 엑셀 시리얼 숫자 판별 (정수이고 40000~50000 범위)
    if pd.api.types.is_numeric_dtype(series):
        numeric_vals = pd.to_numeric(series, errors='coerce')
        if numeric_vals.between(40000, 50000).all():
            return pd.Timestamp('1899-12-30') + pd.to_timedelta(numeric_vals, unit='D')

    s = series.astype(str).str.strip()

    # 8자리 숫자형 판별
    if s.str.match(r'^\d{8}$').all():
        return pd.to_datetime(s, format='%Y%m%d', errors='coerce')

    # 한글형 전처리 → 구분자형으로 변환
    s = s.str.replace(r'년\s*', '-', regex=True)
    s = s.str.replace(r'월\s*', '-', regex=True)
    s = s.str.replace(r'일', '', regex=True)

    return pd.to_datetime(s, infer_datetime_format=True, errors='coerce')


def parse_fisc_month(series):
    """회계월 파싱: '04월' → 4, '4' → 4, '04' → 4"""
    return (
        series.astype(str)
              .str.extract(r'(\d+)')[0]
              .astype('Int64')
    )

In [63]:
# STEP 3 — 날짜 파싱 및 파생변수
date_cfg = cfg['date']

df['post_date'] = parse_date_flexible(df[date_cfg['post_date_col']])
df['doc_date']  = parse_date_flexible(df[date_cfg['doc_date_col']])

# 날짜 파생변수 (_접두사로 원본 컬럼과 구분)
df['_post_year']    = df['post_date'].dt.year
df['_post_month']   = df['post_date'].dt.month
df['_post_day']     = df['post_date'].dt.day
df['_post_weekday'] = df['post_date'].dt.day_name()
df['_post_quarter'] = df['post_date'].dt.quarter
df['_is_weekend']   = df['post_date'].dt.dayofweek >= 5

# 회계월 파싱: '04월' → 4
df['_fisc_month_int'] = parse_fisc_month(df[date_cfg['fisc_month_col']])

parse_errors = df['post_date'].isna().sum()
print(f"날짜 파싱 오류: {parse_errors}건")
if parse_errors > 0:
    print(df[df['post_date'].isna()][['doc_no', 'post_date']].head())

C:\Users\pmpm9\AppData\Local\Temp\ipykernel_24328\1911237982.py:27: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(s, infer_datetime_format=True, errors='coerce')
C:\Users\pmpm9\AppData\Local\Temp\ipykernel_24328\1911237982.py:27: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(s, infer_datetime_format=True, errors='coerce')


날짜 파싱 오류: 0건


In [64]:
# STEP 4 — 금액 표준화 (amount_type 분기)
amt_cfg = cfg['amount']
amount_type = amt_cfg['type']

if amount_type == 'C':
    # C형: 절대금액 컬럼 + 차대구분 플래그
    is_debit = df[amt_cfg['dc_col']].str.contains(
        '|'.join(amt_cfg['debit_keywords']), na=False
    )
    df['_debit']  = df[amt_cfg['amount_col']].where(is_debit, 0)
    df['_credit'] = df[amt_cfg['amount_col']].where(~is_debit, 0)
    df['_amount'] = df['_debit'] - df['_credit']

elif amount_type == 'B':
    # B형: 순금액 단일 컬럼 (양수=차변, 음수=대변)
    df['_amount'] = df[amt_cfg['net_col']]
    df['_debit']  = df['_amount'].clip(lower=0)
    df['_credit'] = (-df['_amount']).clip(lower=0)

elif amount_type == 'A':
    # A형: 차변 컬럼 / 대변 컬럼 분리
    df['_debit']  = df[amt_cfg['debit_col']].fillna(0)
    df['_credit'] = df[amt_cfg['credit_col']].fillna(0)
    df['_amount'] = df['_debit'] - df['_credit']

else:
    raise ValueError(f"지원하지 않는 amount_type: {amount_type} (A/B/C 중 하나여야 함)")

print(f"금액 형태: {amount_type}형")
print(f"차변 건수: {(df['_debit'] > 0).sum():,} / 대변 건수: {(df['_credit'] > 0).sum():,}")

금액 형태: C형
차변 건수: 165,170 / 대변 건수: 166,933


In [65]:
# STEP 5 — 코드 변환 파생변수
flg_cfg = cfg['flags']

df['_is_pl']            = df[flg_cfg['acc_type_col']].str.contains(
                              flg_cfg['acc_type_pl_keyword'], na=False)
df['_is_reversal_flag'] = df[flg_cfg['reversal_col']].str.contains(
                              flg_cfg['reversal_keyword'], na=False)

print(f"PL 계정: {df['_is_pl'].sum():,}건 / BS 계정: {(~df['_is_pl']).sum():,}건")
print(f"역분개:  {df['_is_reversal_flag'].sum():,}건")

PL 계정: 324,742건 / BS 계정: 7,361건
역분개:  5,198건


In [66]:
# STEP 6 — 이상치 플래그
df['_flag_weekend']       = df['_is_weekend']
df['_flag_reversal']      = df['_is_reversal_flag']
df['_flag_out_of_period'] = (
    df['post_date'].dt.year.astype('Int64').astype(str)
    != df[flg_cfg['fisc_year_col']]
)
df['_flag_duplicate']     = df.duplicated(subset=['doc_no', 'line_no'], keep=False)

flag_cols = ['_flag_weekend', '_flag_reversal', '_flag_out_of_period', '_flag_duplicate']
print("이상치 플래그 현황:")
for col in flag_cols:
    print(f"  {col}: {df[col].sum():,}건")

이상치 플래그 현황:
  _flag_weekend: 65,070건
  _flag_reversal: 5,198건
  _flag_out_of_period: 0건
  _flag_duplicate: 322,517건


In [67]:
# 검증
debit_sum  = df['_debit'].sum()
credit_sum = df['_credit'].sum()
diff       = abs(debit_sum - credit_sum)

print("=" * 45)
print(f"분석 대상 행수:  {len(df):,}")
print(f"소계행 분리:     {len(df_subtotal):,}")
print("-" * 45)
print(f"차변 합계: {debit_sum:>22,.2f}")
print(f"대변 합계: {credit_sum:>22,.2f}")
print(f"차이:      {diff:>22,.2f}  {'✅ 균형' if diff < 0.01 else '❌ 불균형'}")
print("=" * 45)
print("\n파생변수 샘플:")
df[[
    'doc_no', 'post_date', '_post_weekday', '_is_weekend', '_fisc_month_int',
    '_debit', '_credit', '_amount',
    '_is_pl', '_is_reversal_flag',
    '_flag_weekend', '_flag_reversal', '_flag_out_of_period', '_flag_duplicate'
]].head()

분석 대상 행수:  332,103
소계행 분리:     0
---------------------------------------------
차변 합계:   2,462,289,614,784.04
대변 합계:   2,462,289,429,380.47
차이:                  185,403.57  ❌ 불균형

파생변수 샘플:


,doc_no,post_date,_post_weekday,_is_weekend,_fisc_month_int,_debit,_credit,_amount,_is_pl,_is_reversal_flag,_flag_weekend,_flag_reversal,_flag_out_of_period,_flag_duplicate
0,5105600151,2022-04-24,Sunday,True,4,300.0,0.0,300.0,True,False,True,False,False,False
1,5105600143,2022-04-24,Sunday,True,4,600.0,0.0,600.0,True,False,True,False,False,False
2,5105600152,2022-04-24,Sunday,True,4,550.0,0.0,550.0,True,False,True,False,False,False
3,5105600153,2022-04-25,Monday,False,4,360.0,0.0,360.0,True,False,False,False,False,False
4,5105600154,2022-04-25,Monday,False,4,480.0,0.0,480.0,True,False,False,False,False,False


In [68]:
# 저장
df.to_csv(cfg['paths']['output_clean'],    index=False, encoding=cfg['encoding'])
df_subtotal.to_csv(cfg['paths']['output_subtotal'], index=False, encoding=cfg['encoding'])

print(f"{cfg['paths']['output_clean']}    저장 완료: {len(df):,}행 / {df.shape[1]}열")
print(f"{cfg['paths']['output_subtotal']} 저장 완료: {len(df_subtotal):,}행")

master_gl_clean.csv    저장 완료: 332,103행 / 34열
master_gl_subtotal.csv 저장 완료: 0행
